In [2]:
import MDAnalysis as mda
from MDAnalysis.analysis import align
from MDAnalysis.tests.datafiles import CRD, PSF, DCD, DCD2
import nglview as nv

import warnings
# suppress some MDAnalysis warnings about writing PDB files
warnings.filterwarnings('ignore')

adk_open = mda.Universe(CRD, DCD2)
adk_closed = mda.Universe(PSF, DCD)
adk_open_view = nv.show_mdanalysis(adk_open)
adk_open_view

NGLWidget(max_frame=101)

In [3]:
adk_closed_view = nv.show_mdanalysis(adk_closed)
adk_closed_view

NGLWidget(max_frame=97)

In [4]:
rmsds = align.alignto(adk_open,  # mobile
                      adk_closed,  # reference
                      select='name CA', # selection to operate on
                      match_atoms=True) # whether to match atoms
print(rmsds)
# (np.float64(21.712154435976014), 6.817293751703919)

aligned_view = nv.show_mdanalysis(mda.Merge(adk_open.atoms, adk_closed.atoms))
aligned_view

(21.781729520578143, 4.695119981939059)


NGLWidget()

In [5]:
rmsds = align.alignto(adk_open.select_atoms('name CA'),  # mobile
                      adk_closed.atoms[:214],  # reference
                      select='all', # selection to operate on
                      match_atoms=False) # whether to match atoms
print(rmsds)

(18.9915652842665, 16.62112485769822)


In [6]:
adk_open = mda.Universe(CRD, DCD2)
adk_closed = mda.Universe(PSF, DCD)

In [7]:
align.AlignTraj(adk_closed,  # trajectory to align
                adk_open,  # reference
                select='name CA',  # selection of atoms to align
                filename='aligned.dcd',  # file to write the trajectory to
                match_atoms=True,  # whether to match atoms based on mass
               ).run()

In [8]:
aligned = mda.Universe(PSF, 'aligned.dcd')
aligned.segments.segids = ['Aligned']  # rename our segments
adk_open.segments.segids = ['Open']  # so they're coloured differently
merged2 = mda.Merge(aligned.atoms, adk_open.atoms)

In [9]:
from MDAnalysis.analysis.base import AnalysisFromFunction
import numpy as np
from MDAnalysis.coordinates.memory import MemoryReader

def copy_coords(ag):
    return ag.positions.copy()

aligned_coords = AnalysisFromFunction(copy_coords,
                                      aligned.atoms).run().results

print(aligned_coords['timeseries'].shape)

(98, 3341, 3)


In [10]:
from nglview.contrib.movie import MovieMaker

You have to install moviepy, imageio and ffmeg
pip install moviepy==0.2.2.11
pip install imageio==1.6


In [11]:
# m2_view = nv.show_mdanalysis(merged2)
# m2_view

NGLWidget()

In [36]:
# from nglview.contrib.movie import MovieMaker
# movie = MovieMaker(
#     m2_view,
#     step=4,  # only render every 4th frame
#     output='merged.gif',
#     render_params={"factor": 3},  # set to 4 for higher quality
# )
# movie.make()

In [37]:
import MDAnalysis as mda
from MDAnalysis.analysis import align, rms
from MDAnalysis.tests.datafiles import PSF, DCD

mobile = mda.Universe(PSF, DCD)
ref = mda.Universe(PSF, DCD)

In [38]:
mobile.trajectory[-1]  # set mobile trajectory to last frame
ref.trajectory[0]  # set reference trajectory to first frame

mobile_ca = mobile.select_atoms('name CA')
ref_ca = ref.select_atoms('name CA')
unaligned_rmsd = rms.rmsd(mobile_ca.positions, ref_ca.positions, superposition=False)
print(f"Unaligned RMSD: {unaligned_rmsd:.2f}")

Unaligned RMSD: 6.84


In [40]:
mobile.trajectory[-1]  # set mobile trajectory to last frame
ref.trajectory[0]  # set reference trajectory to first frame

mobile_ca = mobile.select_atoms('name CA')
ref_ca = ref.select_atoms('name CA')
aligned_rmsd = rms.rmsd(mobile_ca.positions, ref_ca.positions, superposition=False)

print(f"Aligned RMSD: {aligned_rmsd:.2f}")

Aligned RMSD: 6.84
